In [1]:
import numpy as nump
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
import pickle
import timm
import torch.nn as nn
from tqdm import tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
root_folder_path = '/content/'
dataset_zip_path = f'{root_folder_path}drive/MyDrive/MultimodalGasData.zip'

In [4]:
import os
import zipfile

if not os.path.exists(f'{root_folder_path}Gas Sensors Measurements'):
  with zipfile.ZipFile(dataset_zip_path, 'r') as zip_ref:
    zip_ref.extractall(root_folder_path)

In [5]:
file = os.path.join(root_folder_path, 'Thermal Camera Images/')

In [6]:
transformed_training = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(mean = [0.5, 0.5, 0.5], std = [0.5, 0.5, 0.5])])
dataset = datasets.ImageFolder(file, transform = transformed_training)
training_transformed, testing_transformed = random_split(dataset, [int(0.8 * len(dataset)), len(dataset) - int(0.8 * len(dataset))])
training_dataloader = DataLoader(training_transformed, batch_size = 64, shuffle = True)
testing_dataloader = DataLoader(testing_transformed, batch_size = 64, shuffle = False)

In [7]:
model = timm.create_model('deit_base_patch16_224', pretrained = True)
model.head = nn.Linear(model.head.in_features, 4)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f'Training Data Results With Fine Tuning')
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr = 3e-5)
epochs = 7
for epoch in range(epochs):
  model.train()
  loss = 0
  real_prediction = 0
  complete_data = 0
  for batch_images, batch_labels in tqdm(training_dataloader, desc = f'Epoch Number: {epoch + 1}/{epochs}'):
    batch_images = batch_images.to(device)
    batch_labels = batch_labels.to(device)
    optimizer.zero_grad()
    outputs = model(batch_images)
    loss = criterion(outputs, batch_labels)
    loss.backward()
    optimizer.step()
    full_loss = loss + (loss.item() * batch_images.size(0))
    _, predictions = outputs.max(1)
    real_prediction = real_prediction + (predictions == batch_labels).sum().item()
    complete_data = complete_data + batch_labels.size(0)
  accuracy = real_prediction / complete_data
  iteration_loss = full_loss / len(training_dataloader.dataset)
  print(f'Epoch Number: {epoch + 1}, Accuracy: {accuracy:.6f}, Loss: {iteration_loss:.6f}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Training Data Results With Fine Tuning


Epoch Number: 1/7: 100%|██████████| 80/80 [04:14<00:00,  3.18s/it]


Epoch Number: 1, Accuracy: 0.908984, Loss: 0.000573


...

Epoch Number: 5, Accuracy: 0.995898, Loss: 0.000020


Epoch Number: 6/7: 100%|██████████| 80/80 [04:10<00:00,  3.13s/it]


Epoch Number: 6, Accuracy: 0.993750, Loss: 0.000020


Epoch Number: 7/7: 100%|██████████| 80/80 [04:10<00:00,  3.13s/it]

Epoch Number: 7, Accuracy: 0.997852, Loss: 0.000009


In [8]:
print(f'Testing Data Results With Fine Tuning')
model.eval()
real_prediction = 0
complete_test_data = 0
complete_class_test_data = [0] * 4
real_class_prediction = [0] * 4
class_names = ['No Gas', 'Perfume', 'Smoke', 'Mixture']
with torch.no_grad():
  for batch_images, batch_labels in tqdm(testing_dataloader, desc = 'Evaluating'):
    batch_images = batch_images.to(device)
    batch_labels = batch_labels.to(device)
    outputs = model(batch_images)
    _, batch_predictions = outputs.max(1)
    complete_test_data = complete_test_data + batch_labels.size(0)
    real_prediction = real_prediction + (batch_predictions == batch_labels).sum().item()
    for batch_label, batch_prediction in zip(batch_labels, batch_predictions):
      complete_class_test_data[batch_label] = complete_class_test_data[batch_label] + 1
      real_class_prediction[batch_label] = real_class_prediction[batch_label] + (batch_prediction == batch_label).sum().item()
print(f'The Testing Data Accuracy Is: {real_prediction / complete_test_data}')
for class_name, class_real_prediction, class_complete_test_data in zip(class_names, real_class_prediction, complete_class_test_data):
  print(f'The Testing Data Accuracy For The Class {class_name} Is: {class_real_prediction / class_complete_test_data}')

Testing Data Results With Fine Tuning


Evaluating: 100%|██████████| 20/20 [00:36<00:00,  1.84s/it]

The Testing Data Accuracy Is: 0.984375
The Testing Data Accuracy For The Class No Gas Is: 1.0
The Testing Data Accuracy For The Class Perfume Is: 0.9528301886792453
The Testing Data Accuracy For The Class Smoke Is: 0.9837133550488599
The Testing Data Accuracy For The Class Mixture Is: 1.0


In [9]:
torch.save(model.state_dict(), "DeIT_Fine_Tuned_Model.pth")